In [1]:
import os
import gc
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)

# ============================================================
# 1. Select device
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

# Clear unused memory
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# 2. Load pretrained model
# ============================================================

model_name = "distilgpt2"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2 does not have a padding token by default
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True
)

# Explicitly use CPU if CUDA is unavailable
model = model.to(device)

print("Model loaded successfully!")


# ============================================================
# 3. Prepare small training dataset
# ============================================================

texts = [
    "Generative AI creates new content such as text, images, and music.",
    "Machine learning enables computers to learn patterns from data.",
    "Hugging Face provides tools for training and deploying AI models.",
    "Deep learning uses neural networks to solve complex problems.",
    "Natural language processing helps computers understand human language.",
    "Transformers are widely used in modern generative AI applications.",
    "Fine-tuning adapts a pretrained model to a specific task.",
    "Python is a popular programming language for artificial intelligence."
]

# Repeat the small dataset only a few times
texts = texts * 3

dataset = Dataset.from_dict({
    "text": texts
})

print("Dataset size:", len(dataset))


# ============================================================
# 4. Tokenize dataset
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=32
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Dataset tokenized successfully!")


# ============================================================
# 5. Data collator
# ============================================================

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


# ============================================================
# 6. Training configuration
# ============================================================

training_args = TrainingArguments(
    output_dir="./generative_ai_model",

    # Only 1 epoch for practical
    num_train_epochs=1,

    # Very small batch
    per_device_train_batch_size=1,

    # Accumulate gradients instead of increasing batch size
    gradient_accumulation_steps=1,

    # Do not save checkpoints
    save_strategy="no",

    # Logging
    logging_steps=5,

    # Disable external reporting
    report_to="none",

    # CPU training
    use_cpu=(device == "cpu"),

    # Disable unnecessary columns
    remove_unused_columns=True,

    # Disable evaluation
    eval_strategy="no"
)


# ============================================================
# 7. Create Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)


# ============================================================
# 8. Fine-tune model
# ============================================================

print("\nStarting model fine-tuning...")

trainer.train()

print("\nFine-tuning completed!")


# ============================================================
# 9. Save fine-tuned model
# ============================================================

save_path = "./fine_tuned_generative_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("\nModel fine-tuned and saved successfully!")
print("Saved at:", save_path)


# ============================================================
# 10. Create text-generation pipeline
# ============================================================

print("\nLoading text-generation pipeline...")

generator = pipeline(
    "text-generation",
    model=save_path,
    tokenizer=save_path,
    device=0 if device == "cuda" else -1
)


# ============================================================
# 11. Generate text
# ============================================================

prompt = "Generative artificial intelligence"

result = generator(
    prompt,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)


# ============================================================
# 12. Display result
# ============================================================

print("\nGenerated Text:")
print(result[0]["generated_text"])

print("\nDeployment completed successfully!")
print("The fine-tuned model is ready for local text generation.")

Using device: cpu
Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded successfully!
Dataset size: 24


Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Dataset tokenized successfully!

Starting model fine-tuning...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model fine-tuned and saved successfully!
Saved at: ./fine_tuned_generative_model

Loading text-generation pipeline...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]


Generated Text:
Generative artificial intelligence is a new frontier for AI.
























Deployment completed successfully!
The fine-tuned model is ready for local text generation.
